# Warriner Valence Robustness Check

This notebook reruns the frame-aware Sentiment analysis with Warriner et al. norms instead of NRC-VAD. It is a robustness check for the lexicon substitution described in the LSC analysis plan, not a replacement for the main sentiment result.


## Setup

The analysis keeps the main notebook's publication-year axis, frame-aware target strata, separate baseline terms, +/-5-token collocate window, target-term exclusion, and document-level bootstrap. Warriner's native 1-9 valence score is retained, and a 0-1 scaled score is saved for comparison with NRC-VAD trajectories.


In [1]:
from __future__ import annotations

from pathlib import Path
import hashlib
import re

import numpy as np
import pandas as pd
from scipy import stats
import spacy
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 180)


def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "configs/commoncrawl_collection.yaml").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Could not locate repository root from current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd())
CONTEXT_PATH = PROJECT_ROOT / "data/interim/lsc/contexts/lsc_mention_contexts.parquet"
FRAME_LABEL_PATH = PROJECT_ROOT / "data/processed/lsc/classification/lsc_target_context_frame_labels.csv"
WARRINER_PATH = PROJECT_ROOT / "data/external/warriner-norms/warriner_rat.csv"
INTERIM_WARRINER_DIR = PROJECT_ROOT / "data/interim/lsc/warriner_vad"
SENTIMENT_OUTPUT_DIR = PROJECT_ROOT / "data/processed/lsc/sentiment/robustness_warriner"

INTERIM_WARRINER_DIR.mkdir(parents=True, exist_ok=True)
SENTIMENT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WARRINER_MATCHES_PATH = INTERIM_WARRINER_DIR / "lsc_warriner_collocate_matches.parquet"
WARRINER_CONTEXT_COVERAGE_PATH = INTERIM_WARRINER_DIR / "lsc_warriner_context_coverage.parquet"
ANNUAL_VALENCE_PATH = SENTIMENT_OUTPUT_DIR / "lsc_sentiment_warriner_annual_valence.csv"
COVERAGE_PATH = SENTIMENT_OUTPUT_DIR / "lsc_sentiment_warriner_coverage.csv"
TOP_COLLOCATES_PATH = SENTIMENT_OUTPUT_DIR / "lsc_sentiment_warriner_top_collocates.csv"
AUDIT_FLAGS_PATH = SENTIMENT_OUTPUT_DIR / "lsc_sentiment_warriner_audit_flags.csv"
TREND_SUMMARY_PATH = SENTIMENT_OUTPUT_DIR / "lsc_sentiment_warriner_trend_models.csv"
TREND_COMPARISON_PATH = SENTIMENT_OUTPUT_DIR / "lsc_sentiment_warriner_nrc_trend_comparison.csv"
FRAME_CONTEXT_DIAGNOSTICS_PATH = SENTIMENT_OUTPUT_DIR / "lsc_sentiment_warriner_frame_context_diagnostics.csv"

EXPECTED_UNITS = ["ADHD", "Autism", "frustration", "loneliness", "sadness"]
TARGET_UNITS = ["ADHD", "Autism"]
BASELINE_UNITS = ["frustration", "loneliness", "sadness"]
EXPECTED_YEARS = list(range(2014, 2027))
CORE_TARGET_FRAMES = ["clinical_only", "lived_only", "mixed"]
TARGET_FRAME_STRATA = ["substantive_core_overall", *CORE_TARGET_FRAMES]
BASELINE_FRAME_STRATUM = "unframed_baseline"
BOOTSTRAP_REPETITIONS = 500
BOOTSTRAP_SEED = 123
LOW_MATCHED_TOKEN_COVERAGE_WARN = 0.25
LOW_CONTEXT_COVERAGE_WARN = 0.45
TOP_COLLOCATE_SHARE_WARN = 0.20
MIN_FRAME_CONTEXTS_FOR_INTERPRETATION = 100
MIN_FRAME_DOCUMENTS_FOR_INTERPRETATION = 50
DW_AUTOCORRELATION_LOW = 1.25
DW_AUTOCORRELATION_HIGH = 2.75

assert CONTEXT_PATH.exists(), CONTEXT_PATH
assert FRAME_LABEL_PATH.exists(), FRAME_LABEL_PATH
assert WARRINER_PATH.exists(), WARRINER_PATH

try:
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
except OSError as exc:
    raise OSError("Missing spaCy model en_core_web_sm. Install it before running this notebook.") from exc

BAES_EXCLUDED_POS = {"PUNCT", "SYM", "PART", "SPACE", "NUM"}
BAES_EXTRA_STOPWORDS = {"%", "em", "<", ">", "\u03b2", "\u00e0", "\u00d7", "+", "@"}
COLLOCATE_STOPWORDS = set(nlp.Defaults.stop_words) | BAES_EXTRA_STOPWORDS


def normalised_lemma(token: spacy.tokens.Token) -> str:
    lemma = token.lemma_.lower().strip() if token.lemma_ else token.text.lower().strip()
    if lemma == "-pron-":
        lemma = token.text.lower().strip()
    return lemma


def is_baes_collocate_token(token: spacy.tokens.Token) -> bool:
    lemma = normalised_lemma(token)
    lower = token.text.lower().strip()
    if token.pos_ in BAES_EXCLUDED_POS:
        return False
    if token.is_space or token.is_punct or token.like_num:
        return False
    if not token.is_alpha or len(lemma) <= 1:
        return False
    if lower in COLLOCATE_STOPWORDS or lemma in COLLOCATE_STOPWORDS:
        return False
    return True


/opt/anaconda3/envs/msc-nlp/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Inputs And Warriner Norms

Target contexts are expanded into the substantive-core overall stratum and the three core frame strata. Baselines remain unframed and separate. The Warriner file is checked for the summed valence, arousal, and dominance columns used by Baes-style VAD analyses.


In [2]:
context_columns = [
    "doc_id",
    "lsc_year",
    "published_year",
    "source_year",
    "analysis_unit",
    "term_role",
    "target_group",
    "raw_form",
    "matched_text",
    "mention_start_char",
    "mention_end_char",
    "collapsed_raw_forms",
    "collapsed_matched_texts",
    "registered_domain",
    "token_window_5",
]
contexts = pd.read_parquet(CONTEXT_PATH, columns=context_columns).reset_index(drop=True)
contexts["source_context_row_id"] = contexts.index.astype(int)
contexts["target_group"] = contexts["target_group"].fillna("baseline")

observed_units = sorted(contexts["analysis_unit"].dropna().unique())
observed_years = sorted(contexts["lsc_year"].dropna().astype(int).unique())
missing_units = sorted(set(EXPECTED_UNITS) - set(observed_units))
missing_years = sorted(set(EXPECTED_YEARS) - set(observed_years))
if missing_units:
    raise RuntimeError(f"Missing expected analysis units: {missing_units}")
if missing_years:
    raise RuntimeError(f"Missing expected LSC years: {missing_years}")
if contexts["token_window_5"].fillna("").str.strip().eq("").any():
    raise RuntimeError("Some context rows have empty token_window_5 values.")


def stable_context_id(row: pd.Series) -> str:
    value = "|".join(
        str(row.get(column, ""))
        for column in ["doc_id", "analysis_unit", "raw_form", "mention_start_char", "mention_end_char"]
    )
    return hashlib.sha1(value.encode("utf-8")).hexdigest()[:16]


label_columns = [
    "context_id",
    "predicted_derived_frame",
    "p_substantive",
    "p_clinical_given_substantive",
    "p_lived_given_substantive",
]
frame_labels = pd.read_csv(FRAME_LABEL_PATH, usecols=label_columns)
if frame_labels["context_id"].duplicated().any():
    raise RuntimeError("Frame-label handoff contains duplicate context_id values.")

target_mask = contexts["analysis_unit"].isin(TARGET_UNITS)
contexts.loc[target_mask, "context_id"] = contexts.loc[target_mask].apply(stable_context_id, axis=1)
contexts = contexts.merge(frame_labels, on="context_id", how="left")
missing_target_labels = contexts.loc[target_mask, "predicted_derived_frame"].isna().sum()
if missing_target_labels:
    raise RuntimeError(f"Missing frame labels for {missing_target_labels:,} target contexts.")

frame_context_diagnostics = (
    contexts.loc[target_mask]
    .groupby(["analysis_unit", "lsc_year", "predicted_derived_frame"], as_index=False)
    .agg(contexts=("source_context_row_id", "size"), documents=("doc_id", "nunique"))
)
frame_context_diagnostics["included_in_semantic_estimates"] = frame_context_diagnostics[
    "predicted_derived_frame"
].isin(CORE_TARGET_FRAMES)
frame_context_diagnostics["small_cell_flag"] = frame_context_diagnostics[
    "included_in_semantic_estimates"
] & (
    frame_context_diagnostics["contexts"].lt(MIN_FRAME_CONTEXTS_FOR_INTERPRETATION)
    | frame_context_diagnostics["documents"].lt(MIN_FRAME_DOCUMENTS_FOR_INTERPRETATION)
)
frame_context_diagnostics.to_csv(FRAME_CONTEXT_DIAGNOSTICS_PATH, index=False)

baseline_contexts = contexts.loc[~target_mask].copy()
baseline_contexts["frame_stratum"] = BASELINE_FRAME_STRATUM
core_target_contexts = contexts.loc[
    target_mask & contexts["predicted_derived_frame"].isin(CORE_TARGET_FRAMES)
].copy()
target_by_frame = core_target_contexts.copy()
target_by_frame["frame_stratum"] = target_by_frame["predicted_derived_frame"]
target_overall = core_target_contexts.copy()
target_overall["frame_stratum"] = "substantive_core_overall"

analysis_contexts = pd.concat([baseline_contexts, target_overall, target_by_frame], ignore_index=True, sort=False)
analysis_contexts["context_row_id"] = np.arange(len(analysis_contexts), dtype=int)

warriner = pd.read_csv(WARRINER_PATH)
required_warriner_columns = {"word", "V.Mean.Sum", "A.Mean.Sum", "D.Mean.Sum"}
missing_warriner_columns = sorted(required_warriner_columns - set(warriner.columns))
if missing_warriner_columns:
    raise RuntimeError(f"Warriner norms are missing columns: {missing_warriner_columns}")
for column in ["V.Mean.Sum", "A.Mean.Sum", "D.Mean.Sum"]:
    warriner[column] = pd.to_numeric(warriner[column], errors="coerce")
    if not warriner[column].dropna().between(1, 9).all():
        raise RuntimeError(f"Warriner {column} contains values outside the expected 1-9 range.")

warriner = warriner.dropna(subset=["word", "V.Mean.Sum", "A.Mean.Sum", "D.Mean.Sum"]).copy()
warriner["valence_warriner"] = warriner["V.Mean.Sum"].astype(float)
warriner["arousal_warriner"] = warriner["A.Mean.Sum"].astype(float)
warriner["dominance_warriner"] = warriner["D.Mean.Sum"].astype(float)
for source, target in [
    ("valence_warriner", "valence_scaled_0_1"),
    ("arousal_warriner", "arousal_scaled_0_1"),
    ("dominance_warriner", "dominance_scaled_0_1"),
]:
    warriner[target] = (warriner[source] - 1.0) / 8.0

analysis_context_summary = (
    analysis_contexts.groupby(["analysis_unit", "frame_stratum"], as_index=False)
    .agg(contexts=("context_row_id", "size"), documents=("doc_id", "nunique"))
    .sort_values(["analysis_unit", "frame_stratum"])
)
print(f"Original contexts: {len(contexts):,}")
print(f"Analysis context rows after frame expansion: {len(analysis_contexts):,}")
print(f"Warriner norm terms: {len(warriner):,}")
analysis_context_summary


Original contexts: 293,670
Analysis context rows after frame expansion: 311,030
Warriner norm terms: 13,914


,analysis_unit,frame_stratum,contexts,documents
0,ADHD,clinical_only,12039,8006
1,ADHD,lived_only,3610,2730
2,ADHD,mixed,2000,1634
3,ADHD,substantive_core_overall,17649,11444
4,Autism,clinical_only,20306,12959
5,Autism,lived_only,12826,9068
6,Autism,mixed,6331,4735
7,Autism,substantive_core_overall,39463,24141
8,frustration,unframed_baseline,105482,93663
9,loneliness,unframed_baseline,38052,30431


## Prepare Warriner Lookup

Warriner terms are normalised to lowercase alphabetic keys. The resource is mostly unigram-based, but the matching code supports multi-token keys so that the lookup remains explicit and auditable.


In [3]:
def normalise_lexicon_key(term: object) -> tuple[str, ...]:
    doc = nlp(str(term).replace("_", " "))
    if any(not is_baes_collocate_token(token) for token in doc):
        return ()
    return tuple(normalised_lemma(token) for token in doc)


lexicon_rows: dict[tuple[str, ...], list[dict[str, object]]] = {}
dropped_lexicon_entries = 0
for row in warriner.itertuples(index=False):
    key = normalise_lexicon_key(row.word)
    if not key:
        dropped_lexicon_entries += 1
        continue
    lexicon_rows.setdefault(key, []).append(
        {
            "word": row.word,
            "valence_warriner": float(row.valence_warriner),
            "arousal_warriner": float(row.arousal_warriner),
            "dominance_warriner": float(row.dominance_warriner),
            "valence_scaled_0_1": float(row.valence_scaled_0_1),
            "arousal_scaled_0_1": float(row.arousal_scaled_0_1),
            "dominance_scaled_0_1": float(row.dominance_scaled_0_1),
        }
    )

lexicon_lookup: dict[tuple[str, ...], dict[str, object]] = {}
for key, rows in lexicon_rows.items():
    lexicon_lookup[key] = {
        "lexicon_key": " ".join(key),
        "source_terms": "|".join(sorted({str(row["word"]) for row in rows})),
        "source_term_count": len(rows),
        "valence_warriner": float(np.mean([row["valence_warriner"] for row in rows])),
        "arousal_warriner": float(np.mean([row["arousal_warriner"] for row in rows])),
        "dominance_warriner": float(np.mean([row["dominance_warriner"] for row in rows])),
        "valence_scaled_0_1": float(np.mean([row["valence_scaled_0_1"] for row in rows])),
        "arousal_scaled_0_1": float(np.mean([row["arousal_scaled_0_1"] for row in rows])),
        "dominance_scaled_0_1": float(np.mean([row["dominance_scaled_0_1"] for row in rows])),
    }

key_lengths = sorted({len(key) for key in lexicon_lookup}, reverse=True)
print(f"Normalised Warriner keys: {len(lexicon_lookup):,}")
print(f"Dropped Warriner entries during Baes-style normalisation: {dropped_lexicon_entries:,}")
print(f"Longest matched key length: {max(key_lengths)}")


Normalised Warriner keys: 13,309
Dropped Warriner entries during Baes-style normalisation: 105
Longest matched key length: 3


## Extract Warriner-Matched Collocates

The extraction mirrors the main NRC-VAD collocate handoff: lemmatise local windows, remove the focal target or baseline term, skip punctuation/symbol/space/number/particle tokens, remove stopwords, and greedily match Warriner keys.


In [4]:
TARGET_RAW_FORM_EXCLUSION_TOKENS = {
    "adhd": {"adhd"},
    "attention_deficit": {"attention", "deficit", "hyperactivity", "disorder"},
    "autism": {"autism"},
    "autistic": {"autistic"},
    "autism_spectrum": {"autism", "spectrum"},
    "asd_disambiguated": {"asd"},
}
BASELINE_RAW_FORM_EXCLUSION_TOKENS = {
    "frustration": {"frustration"},
    "sadness": {"sadness"},
    "loneliness": {"loneliness"},
}
RAW_FORM_EXCLUSION_TOKENS_BY_ROLE = {
    "target": TARGET_RAW_FORM_EXCLUSION_TOKENS,
    "baseline": BASELINE_RAW_FORM_EXCLUSION_TOKENS,
}


def split_pipe_values(value: object) -> list[str]:
    if pd.isna(value):
        return []
    return [part.strip() for part in str(value).split("|") if part.strip()]


def exclusion_tokens_for_row(row: pd.Series) -> set[str]:
    raw_forms = {str(row["raw_form"])} | set(split_pipe_values(row.get("collapsed_raw_forms")))
    role_exclusions = RAW_FORM_EXCLUSION_TOKENS_BY_ROLE.get(str(row["term_role"]), {})
    tokens: set[str] = set()
    for raw_form in raw_forms:
        tokens.update(role_exclusions.get(raw_form, {raw_form.replace("_", " ")}))
    matched_text = str(row.get("matched_text") or "")
    if matched_text:
        tokens.update(part.lower() for part in re.findall(r"[A-Za-z]+", matched_text))
    return tokens


def lexical_tokens(doc, excluded_terms: set[str]) -> list[dict[str, object]]:
    tokens = []
    for token in doc:
        lemma = normalised_lemma(token)
        lower = token.text.lower()
        if not is_baes_collocate_token(token):
            continue
        if lemma in excluded_terms or lower in excluded_terms:
            continue
        tokens.append({"token_index": token.i, "text": token.text, "lemma": lemma})
    return tokens


def match_warriner_units(tokens: list[dict[str, object]]) -> list[dict[str, object]]:
    matches: list[dict[str, object]] = []
    index = 0
    while index < len(tokens):
        matched = None
        for length in key_lengths:
            if index + length > len(tokens):
                continue
            key = tuple(token["lemma"] for token in tokens[index : index + length])
            if key in lexicon_lookup:
                matched = (length, lexicon_lookup[key], "phrase" if length > 1 else "unigram")
                break
        if matched is None:
            index += 1
            continue
        length, lexicon_row, collocate_type = matched
        span_tokens = tokens[index : index + length]
        matches.append(
            {
                "collocate": lexicon_row["lexicon_key"],
                "collocate_type": collocate_type,
                "surface_text": " ".join(str(token["text"]) for token in span_tokens),
                "token_start_in_window": int(span_tokens[0]["token_index"]),
                "token_end_in_window": int(span_tokens[-1]["token_index"] + 1),
                "matched_token_count": int(length),
                "valence_warriner": float(lexicon_row["valence_warriner"]),
                "arousal_warriner": float(lexicon_row["arousal_warriner"]),
                "dominance_warriner": float(lexicon_row["dominance_warriner"]),
                "valence_scaled_0_1": float(lexicon_row["valence_scaled_0_1"]),
                "arousal_scaled_0_1": float(lexicon_row["arousal_scaled_0_1"]),
                "dominance_scaled_0_1": float(lexicon_row["dominance_scaled_0_1"]),
                "source_terms": lexicon_row["source_terms"],
                "source_term_count": int(lexicon_row["source_term_count"]),
            }
        )
        index += length
    return matches


match_rows: list[dict[str, object]] = []
coverage_rows: list[dict[str, object]] = []

for doc, (_, row) in zip(
    nlp.pipe(analysis_contexts["token_window_5"].astype(str), batch_size=1000),
    analysis_contexts.iterrows(),
):
    excluded_terms = exclusion_tokens_for_row(row)
    tokens = lexical_tokens(doc, excluded_terms)
    row_matches = match_warriner_units(tokens)
    matched_token_positions = sum(match["matched_token_count"] for match in row_matches)

    coverage_rows.append(
        {
            "context_row_id": int(row["context_row_id"]),
            "source_context_row_id": int(row["source_context_row_id"]),
            "context_id": row.get("context_id"),
            "doc_id": row["doc_id"],
            "lsc_year": int(row["lsc_year"]),
            "analysis_unit": row["analysis_unit"],
            "term_role": row["term_role"],
            "target_group": row["target_group"],
            "raw_form": row["raw_form"],
            "frame_stratum": row["frame_stratum"],
            "predicted_derived_frame": row.get("predicted_derived_frame"),
            "candidate_collocate_tokens": len(tokens),
            "matched_warriner_units": len(row_matches),
            "matched_token_positions": int(matched_token_positions),
            "has_warriner_match": bool(row_matches),
        }
    )

    for match in row_matches:
        match_rows.append(
            {
                "context_row_id": int(row["context_row_id"]),
                "source_context_row_id": int(row["source_context_row_id"]),
                "context_id": row.get("context_id"),
                "doc_id": row["doc_id"],
                "lsc_year": int(row["lsc_year"]),
                "analysis_unit": row["analysis_unit"],
                "term_role": row["term_role"],
                "target_group": row["target_group"],
                "raw_form": row["raw_form"],
                "frame_stratum": row["frame_stratum"],
                "predicted_derived_frame": row.get("predicted_derived_frame"),
                **match,
            }
        )

warriner_matches = pd.DataFrame(match_rows)
context_coverage = pd.DataFrame(coverage_rows)
if warriner_matches.empty:
    raise RuntimeError("No Warriner collocates matched the frame-aware LSC contexts.")
stopword_collocate_rows = warriner_matches["collocate"].map(lambda value: any(part in COLLOCATE_STOPWORDS for part in str(value).split())).sum()
if stopword_collocate_rows:
    raise RuntimeError(f"Stopword collocates survived Baes-style filtering: {int(stopword_collocate_rows):,}")

warriner_matches.to_parquet(WARRINER_MATCHES_PATH, index=False)
context_coverage.to_parquet(WARRINER_CONTEXT_COVERAGE_PATH, index=False)
print(f"Warriner match rows: {len(warriner_matches):,}")
print(f"Contexts with any Warriner match: {context_coverage['has_warriner_match'].sum():,} / {len(context_coverage):,}")


Warriner match rows: 990,407
Contexts with any Warriner match: 306,644 / 311,030


## Annual Valence Index

Annual scores are frequency-weighted means over matched collocates. Coverage is saved separately so that any divergence from the NRC-VAD result can be interpreted alongside lexicon coverage.


In [5]:
GROUP_COLUMNS = ["lsc_year", "analysis_unit", "frame_stratum"]

coverage = (
    context_coverage.groupby(GROUP_COLUMNS, as_index=False)
    .agg(
        context_rows=("context_row_id", "size"),
        documents=("doc_id", "nunique"),
        candidate_collocate_tokens=("candidate_collocate_tokens", "sum"),
        matched_warriner_units_coverage=("matched_warriner_units", "sum"),
        matched_token_positions=("matched_token_positions", "sum"),
        contexts_with_warriner_match=("has_warriner_match", "sum"),
    )
)
coverage["matched_token_coverage"] = coverage["matched_token_positions"] / coverage["candidate_collocate_tokens"].replace(0, np.nan)
coverage["context_match_coverage"] = coverage["contexts_with_warriner_match"] / coverage["context_rows"].replace(0, np.nan)
coverage["small_cell_flag"] = coverage["frame_stratum"].isin(CORE_TARGET_FRAMES) & (
    coverage["context_rows"].lt(MIN_FRAME_CONTEXTS_FOR_INTERPRETATION)
    | coverage["documents"].lt(MIN_FRAME_DOCUMENTS_FOR_INTERPRETATION)
)

annual_valence = (
    warriner_matches.groupby([*GROUP_COLUMNS, "term_role", "target_group"], as_index=False)
    .agg(
        valence_warriner_mean=("valence_warriner", "mean"),
        valence_warriner_sd=("valence_warriner", "std"),
        valence_scaled_mean=("valence_scaled_0_1", "mean"),
        valence_scaled_sd=("valence_scaled_0_1", "std"),
        arousal_warriner_mean_for_reuse=("arousal_warriner", "mean"),
        arousal_scaled_mean_for_reuse=("arousal_scaled_0_1", "mean"),
        dominance_warriner_mean_for_reuse=("dominance_warriner", "mean"),
        matched_warriner_units=("valence_warriner", "size"),
        unique_collocates=("collocate", "nunique"),
        documents_with_matches=("doc_id", "nunique"),
    )
)
annual_valence = annual_valence.merge(coverage, on=GROUP_COLUMNS, how="left")
annual_valence = annual_valence.sort_values(["analysis_unit", "frame_stratum", "lsc_year"]).reset_index(drop=True)
annual_valence.head(12)


,lsc_year,analysis_unit,frame_stratum,term_role,target_group,valence_warriner_mean,valence_warriner_sd,valence_scaled_mean,valence_scaled_sd,arousal_warriner_mean_for_reuse,arousal_scaled_mean_for_reuse,dominance_warriner_mean_for_reuse,matched_warriner_units,unique_collocates,documents_with_matches,context_rows,documents,candidate_collocate_tokens,matched_warriner_units_coverage,matched_token_positions,contexts_with_warriner_match,matched_token_coverage,context_match_coverage,small_cell_flag
0,2014,ADHD,clinical_only,target,ADHD,5.253636,1.385688,0.531705,0.173211,4.252283,0.406535,5.301367,4024,925,784,1286,808,4888,4024,4026,1235,0.823650,0.960342,False
1,2015,ADHD,clinical_only,target,ADHD,5.254152,1.391470,0.531769,0.173934,4.229486,0.403686,5.330384,3617,909,753,1203,774,4464,3617,3619,1158,0.810708,0.962594,False
2,2016,ADHD,clinical_only,target,ADHD,5.175759,1.464569,0.521970,0.183071,4.230597,0.403825,5.277579,3581,907,776,1149,790,4295,3581,3582,1114,0.833993,0.969539,False
3,2017,ADHD,clinical_only,target,ADHD,5.182436,1.443453,0.522805,0.180432,4.232982,0.404123,5.260890,3358,903,730,1099,751,4123,3358,3363,1049,0.815668,0.954504,False
4,2018,ADHD,clinical_only,target,ADHD,5.229120,1.458496,0.528640,0.182312,4.229745,0.403718,5.319442,3782,931,780,1182,787,4602,3782,3787,1159,0.822903,0.980541,False
5,2019,ADHD,clinical_only,target,ADHD,5.122672,1.493860,0.515334,0.186732,4.240207,0.405026,5.261324,3214,812,658,1000,674,3957,3214,3214,969,0.812231,0.969000,False
6,2020,ADHD,clinical_only,target,ADHD,5.166083,1.471067,0.520760,0.183883,4.223875,0.402984,5.269291,2912,761,659,959,668,3590,2912,2915,924,0.811978,0.963504,False
7,2021,ADHD,clinical_only,target,ADHD,5.141000,1.490238,0.517625,0.186280,4.250567,0.406321,5.221936,2917,845,630,928,636,3565,2917,2920,903,0.819074,0.973060,False
8,2022,ADHD,clinical_only,target,ADHD,5.216311,1.456086,0.527039,0.182011,4.195264,0.399408,5.341959,3006,811,623,930,634,3606,3006,3007,903,0.833888,0.970968,False
9,2023,ADHD,clinical_only,target,ADHD,5.210044,1.425183,0.526256,0.178148,4.224359,0.403045,5.277719,2307,711,467,750,477,2845,2307,2308,721,0.811248,0.961333,False


## Bootstrap Confidence Intervals

Document-level bootstrap intervals keep the uncertainty unit aligned with the main sentiment notebook.


In [6]:
rng = np.random.default_rng(BOOTSTRAP_SEED)
bootstrap_rows: list[dict[str, object]] = []

doc_scores = (
    warriner_matches.groupby([*GROUP_COLUMNS, "doc_id"], as_index=False)
    .agg(
        valence_warriner_sum=("valence_warriner", "sum"),
        valence_scaled_sum=("valence_scaled_0_1", "sum"),
        matched_warriner_units=("valence_warriner", "size"),
    )
)

for group_values, doc_frame in doc_scores.groupby(GROUP_COLUMNS, sort=True):
    native_sums = doc_frame["valence_warriner_sum"].to_numpy(dtype=float)
    scaled_sums = doc_frame["valence_scaled_sum"].to_numpy(dtype=float)
    unit_counts = doc_frame["matched_warriner_units"].to_numpy(dtype=float)
    n_docs = len(doc_frame)
    native_estimates = np.empty(BOOTSTRAP_REPETITIONS, dtype=float)
    scaled_estimates = np.empty(BOOTSTRAP_REPETITIONS, dtype=float)
    for repetition in range(BOOTSTRAP_REPETITIONS):
        sample_index = rng.integers(0, n_docs, size=n_docs)
        denominator = unit_counts[sample_index].sum()
        native_estimates[repetition] = native_sums[sample_index].sum() / denominator if denominator else np.nan
        scaled_estimates[repetition] = scaled_sums[sample_index].sum() / denominator if denominator else np.nan
    bootstrap_rows.append(
        {
            "lsc_year": int(group_values[0]),
            "analysis_unit": group_values[1],
            "frame_stratum": group_values[2],
            "bootstrap_repetitions": BOOTSTRAP_REPETITIONS,
            "bootstrap_unit": "doc_id",
            "valence_warriner_bootstrap_mean": float(np.nanmean(native_estimates)),
            "valence_warriner_ci_low": float(np.nanpercentile(native_estimates, 2.5)),
            "valence_warriner_ci_high": float(np.nanpercentile(native_estimates, 97.5)),
            "valence_scaled_bootstrap_mean": float(np.nanmean(scaled_estimates)),
            "valence_scaled_ci_low": float(np.nanpercentile(scaled_estimates, 2.5)),
            "valence_scaled_ci_high": float(np.nanpercentile(scaled_estimates, 97.5)),
        }
    )

bootstrap = pd.DataFrame(bootstrap_rows)
annual_valence = annual_valence.merge(bootstrap, on=GROUP_COLUMNS, how="left")
annual_valence.to_csv(ANNUAL_VALENCE_PATH, index=False)
coverage.to_csv(COVERAGE_PATH, index=False)
annual_valence.head(12)


,lsc_year,analysis_unit,frame_stratum,term_role,target_group,valence_warriner_mean,valence_warriner_sd,valence_scaled_mean,valence_scaled_sd,arousal_warriner_mean_for_reuse,arousal_scaled_mean_for_reuse,dominance_warriner_mean_for_reuse,matched_warriner_units,unique_collocates,documents_with_matches,context_rows,documents,candidate_collocate_tokens,matched_warriner_units_coverage,matched_token_positions,contexts_with_warriner_match,matched_token_coverage,context_match_coverage,small_cell_flag,bootstrap_repetitions,bootstrap_unit,valence_warriner_bootstrap_mean,valence_warriner_ci_low,valence_warriner_ci_high,valence_scaled_bootstrap_mean,valence_scaled_ci_low,valence_scaled_ci_high
0,2014,ADHD,clinical_only,target,ADHD,5.253636,1.385688,0.531705,0.173211,4.252283,0.406535,5.301367,4024,925,784,1286,808,4888,4024,4026,1235,0.823650,0.960342,False,500,doc_id,5.252889,5.197391,5.301904,0.531611,0.524674,0.537738
1,2015,ADHD,clinical_only,target,ADHD,5.254152,1.391470,0.531769,0.173934,4.229486,0.403686,5.330384,3617,909,753,1203,774,4464,3617,3619,1158,0.810708,0.962594,False,500,doc_id,5.254430,5.199418,5.306147,0.531804,0.524927,0.538268
2,2016,ADHD,clinical_only,target,ADHD,5.175759,1.464569,0.521970,0.183071,4.230597,0.403825,5.277579,3581,907,776,1149,790,4295,3581,3582,1114,0.833993,0.969539,False,500,doc_id,5.174987,5.114374,5.235482,0.521873,0.514297,0.529435
3,2017,ADHD,clinical_only,target,ADHD,5.182436,1.443453,0.522805,0.180432,4.232982,0.404123,5.260890,3358,903,730,1099,751,4123,3358,3363,1049,0.815668,0.954504,False,500,doc_id,5.180798,5.121333,5.246289,0.522600,0.515167,0.530786
4,2018,ADHD,clinical_only,target,ADHD,5.229120,1.458496,0.528640,0.182312,4.229745,0.403718,5.319442,3782,931,780,1182,787,4602,3782,3787,1159,0.822903,0.980541,False,500,doc_id,5.228614,5.170632,5.285604,0.528577,0.521329,0.535700
5,2019,ADHD,clinical_only,target,ADHD,5.122672,1.493860,0.515334,0.186732,4.240207,0.405026,5.261324,3214,812,658,1000,674,3957,3214,3214,969,0.812231,0.969000,False,500,doc_id,5.120185,5.054300,5.190653,0.515023,0.506787,0.523832
6,2020,ADHD,clinical_only,target,ADHD,5.166083,1.471067,0.520760,0.183883,4.223875,0.402984,5.269291,2912,761,659,959,668,3590,2912,2915,924,0.811978,0.963504,False,500,doc_id,5.165553,5.098852,5.232815,0.520694,0.512356,0.529102
7,2021,ADHD,clinical_only,target,ADHD,5.141000,1.490238,0.517625,0.186280,4.250567,0.406321,5.221936,2917,845,630,928,636,3565,2917,2920,903,0.819074,0.973060,False,500,doc_id,5.142827,5.076238,5.211240,0.517853,0.509530,0.526405
8,2022,ADHD,clinical_only,target,ADHD,5.216311,1.456086,0.527039,0.182011,4.195264,0.399408,5.341959,3006,811,623,930,634,3606,3006,3007,903,0.833888,0.970968,False,500,doc_id,5.216488,5.156332,5.276555,0.527061,0.519541,0.534569
9,2023,ADHD,clinical_only,target,ADHD,5.210044,1.425183,0.526256,0.178148,4.224359,0.403045,5.277719,2307,711,467,750,477,2845,2307,2308,721,0.811248,0.961333,False,500,doc_id,5.209523,5.140934,5.276701,0.526190,0.517617,0.534588


## Trend Models

The notebook fits the same compact OLS trend summaries as the main scalar LSC analyses. The scaled Warriner trend is also merged with the main NRC-VAD trend table for direction checks.


In [7]:
def fit_trend(frame: pd.DataFrame, value_column: str) -> dict[str, object]:
    data = frame[["lsc_year", value_column]].dropna().sort_values("lsc_year")
    if len(data) < 3 or data[value_column].nunique() < 2:
        return {
            "n_years": len(data),
            "year_center": np.nan,
            "linear_intercept": np.nan,
            "linear_slope_per_year": np.nan,
            "linear_slope_se": np.nan,
            "linear_p_value": np.nan,
            "linear_r_squared": np.nan,
            "linear_adj_r_squared": np.nan,
            "standardized_beta_year": np.nan,
            "durbin_watson": np.nan,
            "lag1_residual_autocorrelation": np.nan,
            "autocorrelation_flag": False,
            "ar1_sensitivity_slope_per_year": np.nan,
            "ar1_sensitivity_p_value": np.nan,
            "quadratic_adj_r_squared": np.nan,
            "quadratic_delta_adj_r_squared": np.nan,
        }
    years = data["lsc_year"].to_numpy(dtype=float)
    values = data[value_column].to_numpy(dtype=float)
    year_center = float(years.mean())
    x = years - year_center
    result = stats.linregress(x, values)
    fitted = result.intercept + result.slope * x
    residuals = values - fitted
    sse = float(np.sum(residuals**2))
    sst = float(np.sum((values - values.mean()) ** 2))
    r_squared = 1 - sse / sst if sst else np.nan
    n = len(values)
    adj_r_squared = 1 - (1 - r_squared) * (n - 1) / (n - 2) if n > 2 and not np.isnan(r_squared) else np.nan
    std_beta = result.slope * np.std(x, ddof=1) / np.std(values, ddof=1) if np.std(values, ddof=1) else np.nan
    dw_denominator = float(np.sum(residuals**2))
    durbin_watson = float(np.sum(np.diff(residuals) ** 2) / dw_denominator) if dw_denominator else np.nan
    lag1 = float(np.corrcoef(residuals[:-1], residuals[1:])[0, 1]) if n >= 4 and np.std(residuals) else np.nan
    autocorrelation_flag = bool(
        not np.isnan(durbin_watson)
        and (durbin_watson < DW_AUTOCORRELATION_LOW or durbin_watson > DW_AUTOCORRELATION_HIGH)
    )
    ar1_slope = np.nan
    ar1_p_value = np.nan
    if autocorrelation_flag and n >= 5 and not np.isnan(lag1) and abs(lag1) < 0.98:
        y_star = values[1:] - lag1 * values[:-1]
        x_star = x[1:] - lag1 * x[:-1]
        ar1_result = stats.linregress(x_star, y_star)
        ar1_slope = float(ar1_result.slope)
        ar1_p_value = float(ar1_result.pvalue)
    quadratic_adj_r_squared = np.nan
    quadratic_delta = np.nan
    if n >= 6:
        q_coefficients = np.polyfit(x, values, deg=2)
        q_fitted = np.polyval(q_coefficients, x)
        q_sse = float(np.sum((values - q_fitted) ** 2))
        q_r_squared = 1 - q_sse / sst if sst else np.nan
        quadratic_adj_r_squared = 1 - (1 - q_r_squared) * (n - 1) / (n - 3) if n > 3 and not np.isnan(q_r_squared) else np.nan
        quadratic_delta = quadratic_adj_r_squared - adj_r_squared if not np.isnan(quadratic_adj_r_squared) else np.nan
    return {
        "n_years": n,
        "year_center": year_center,
        "linear_intercept": float(result.intercept),
        "linear_slope_per_year": float(result.slope),
        "linear_slope_se": float(result.stderr) if result.stderr is not None else np.nan,
        "linear_p_value": float(result.pvalue),
        "linear_r_squared": float(r_squared),
        "linear_adj_r_squared": float(adj_r_squared),
        "standardized_beta_year": float(std_beta),
        "durbin_watson": durbin_watson,
        "lag1_residual_autocorrelation": lag1,
        "autocorrelation_flag": autocorrelation_flag,
        "ar1_sensitivity_slope_per_year": ar1_slope,
        "ar1_sensitivity_p_value": ar1_p_value,
        "quadratic_adj_r_squared": quadratic_adj_r_squared,
        "quadratic_delta_adj_r_squared": quadratic_delta,
    }


trend_rows = []
index_columns = {
    "warriner_valence_native_1_9": "valence_warriner_mean",
    "warriner_valence_scaled_0_1": "valence_scaled_mean",
}
for group_values, frame in annual_valence.groupby(["analysis_unit", "frame_stratum", "term_role", "target_group"], sort=True):
    analysis_unit, frame_stratum, term_role, target_group = group_values
    for index_name, value_column in index_columns.items():
        trend_rows.append(
            {
                "analysis_unit": analysis_unit,
                "frame_stratum": frame_stratum,
                "term_role": term_role,
                "target_group": target_group,
                "index_name": index_name,
                "value_column": value_column,
                **fit_trend(frame, value_column),
            }
        )
trend_summary = pd.DataFrame(trend_rows)
trend_summary.to_csv(TREND_SUMMARY_PATH, index=False)

main_trend_path = PROJECT_ROOT / "data/processed/lsc/sentiment/lsc_sentiment_trend_models.csv"
if main_trend_path.exists():
    main_trends = pd.read_csv(main_trend_path)
    main_subset = main_trends.loc[main_trends["index_name"].eq("valence_mean")].copy()
    robust_subset = trend_summary.loc[trend_summary["index_name"].eq("warriner_valence_scaled_0_1")].copy()
    comparison = robust_subset.merge(
        main_subset[
            [
                "analysis_unit",
                "frame_stratum",
                "linear_slope_per_year",
                "linear_p_value",
                "linear_adj_r_squared",
                "autocorrelation_flag",
            ]
        ].rename(
            columns={
                "linear_slope_per_year": "nrc_linear_slope_per_year",
                "linear_p_value": "nrc_linear_p_value",
                "linear_adj_r_squared": "nrc_linear_adj_r_squared",
                "autocorrelation_flag": "nrc_autocorrelation_flag",
            }
        ),
        on=["analysis_unit", "frame_stratum"],
        how="left",
    )
    comparison = comparison.rename(
        columns={
            "linear_slope_per_year": "warriner_scaled_linear_slope_per_year",
            "linear_p_value": "warriner_scaled_linear_p_value",
            "linear_adj_r_squared": "warriner_scaled_linear_adj_r_squared",
            "autocorrelation_flag": "warriner_scaled_autocorrelation_flag",
        }
    )
    comparison["slope_direction_agrees"] = np.sign(comparison["warriner_scaled_linear_slope_per_year"]) == np.sign(
        comparison["nrc_linear_slope_per_year"]
    )
    comparison.to_csv(TREND_COMPARISON_PATH, index=False)
else:
    comparison = pd.DataFrame()

trend_summary.head(12)


,analysis_unit,frame_stratum,term_role,target_group,index_name,value_column,n_years,year_center,linear_intercept,linear_slope_per_year,linear_slope_se,linear_p_value,linear_r_squared,linear_adj_r_squared,standardized_beta_year,durbin_watson,lag1_residual_autocorrelation,autocorrelation_flag,ar1_sensitivity_slope_per_year,ar1_sensitivity_p_value,quadratic_adj_r_squared,quadratic_delta_adj_r_squared
0,ADHD,clinical_only,target,ADHD,warriner_valence_native_1_9,valence_warriner_mean,13,2020.0,5.198315,-0.001446,0.003059,0.645520,0.019926,-0.069171,-0.141161,1.524178,0.155098,False,NaN,NaN,0.352378,0.421549
1,ADHD,clinical_only,target,ADHD,warriner_valence_scaled_0_1,valence_scaled_mean,13,2020.0,0.524789,-0.000181,0.000382,0.645520,0.019926,-0.069171,-0.141161,1.524178,0.155098,False,NaN,NaN,0.352378,0.421549
2,ADHD,lived_only,target,ADHD,warriner_valence_native_1_9,valence_warriner_mean,13,2020.0,5.561861,-0.003920,0.006242,0.542811,0.034614,-0.053148,-0.186048,1.654151,-0.246222,False,NaN,NaN,0.069426,0.122574
3,ADHD,lived_only,target,ADHD,warriner_valence_scaled_0_1,valence_scaled_mean,13,2020.0,0.570233,-0.000490,0.000780,0.542811,0.034614,-0.053148,-0.186048,1.654151,-0.246222,False,NaN,NaN,0.069426,0.122574
4,ADHD,mixed,target,ADHD,warriner_valence_native_1_9,valence_warriner_mean,13,2020.0,5.408571,0.012579,0.007581,0.125266,0.200190,0.127480,0.447426,2.137324,-0.243978,False,NaN,NaN,0.267770,0.140290
5,ADHD,mixed,target,ADHD,warriner_valence_scaled_0_1,valence_scaled_mean,13,2020.0,0.551071,0.001572,0.000948,0.125266,0.200190,0.127480,0.447426,2.137324,-0.243978,False,NaN,NaN,0.267770,0.140290
6,ADHD,substantive_core_overall,target,ADHD,warriner_valence_native_1_9,valence_warriner_mean,13,2020.0,5.294180,0.002767,0.002938,0.366445,0.074640,-0.009483,0.273204,1.361246,0.286955,False,NaN,NaN,0.046364,0.055847
7,ADHD,substantive_core_overall,target,ADHD,warriner_valence_scaled_0_1,valence_scaled_mean,13,2020.0,0.536772,0.000346,0.000367,0.366445,0.074640,-0.009483,0.273204,1.361246,0.286955,False,NaN,NaN,0.046364,0.055847
8,Autism,clinical_only,target,Autism,warriner_valence_native_1_9,valence_warriner_mean,13,2020.0,5.369769,0.006612,0.003383,0.076578,0.257689,0.190206,0.507630,0.947734,0.430608,True,0.014263,0.012362,0.746009,0.555804
9,Autism,clinical_only,target,Autism,warriner_valence_scaled_0_1,valence_scaled_mean,13,2020.0,0.546221,0.000826,0.000423,0.076578,0.257689,0.190206,0.507630,0.947734,0.430608,True,0.001783,0.012362,0.746009,0.555804


## Diagnostics And Saved Outputs

Top collocates and audit flags are retained as CSV diagnostics rather than report figures.


In [8]:
collocate_counts = (
    warriner_matches.groupby([*GROUP_COLUMNS, "collocate"], as_index=False)
    .agg(collocate_count=("collocate", "size"))
)
collocate_counts["group_match_count"] = collocate_counts.groupby(GROUP_COLUMNS)["collocate_count"].transform("sum")
collocate_counts["collocate_share"] = collocate_counts["collocate_count"] / collocate_counts["group_match_count"]
top_share = collocate_counts.sort_values("collocate_share", ascending=False).groupby(GROUP_COLUMNS, as_index=False).head(1)
top_share = top_share.rename(columns={"collocate": "top_collocate", "collocate_share": "top_collocate_share"})[
    [*GROUP_COLUMNS, "top_collocate", "top_collocate_share"]
]

top_collocates = (
    warriner_matches.groupby(["analysis_unit", "frame_stratum", "collocate"], as_index=False)
    .agg(
        occurrences=("collocate", "size"),
        documents=("doc_id", "nunique"),
        valence_warriner_mean=("valence_warriner", "mean"),
        valence_scaled_mean=("valence_scaled_0_1", "mean"),
        arousal_warriner_mean=("arousal_warriner", "mean"),
        source_terms=("source_terms", "first"),
    )
)
top_collocates["unit_frame_occurrences"] = top_collocates.groupby(["analysis_unit", "frame_stratum"])["occurrences"].transform("sum")
top_collocates["collocate_share"] = top_collocates["occurrences"] / top_collocates["unit_frame_occurrences"]
top_collocates = (
    top_collocates.sort_values(["analysis_unit", "frame_stratum", "occurrences"], ascending=[True, True, False])
    .groupby(["analysis_unit", "frame_stratum"], as_index=False)
    .head(25)
    .reset_index(drop=True)
)
top_collocates.to_csv(TOP_COLLOCATES_PATH, index=False)

coverage_for_flags = coverage.merge(top_share, on=GROUP_COLUMNS, how="left")
flag_rows = []
for row in coverage_for_flags.itertuples(index=False):
    flags = []
    if bool(row.small_cell_flag):
        flags.append("small_frame_year_cell")
    if pd.notna(row.matched_token_coverage) and row.matched_token_coverage < LOW_MATCHED_TOKEN_COVERAGE_WARN:
        flags.append("low_matched_token_coverage")
    if pd.notna(row.context_match_coverage) and row.context_match_coverage < LOW_CONTEXT_COVERAGE_WARN:
        flags.append("low_context_match_coverage")
    if pd.notna(row.top_collocate_share) and row.top_collocate_share > TOP_COLLOCATE_SHARE_WARN:
        flags.append("dominant_top_collocate")
    if flags:
        flag_rows.append(
            {
                "lsc_year": row.lsc_year,
                "analysis_unit": row.analysis_unit,
                "frame_stratum": row.frame_stratum,
                "flags": ";".join(flags),
                "context_rows": row.context_rows,
                "documents": row.documents,
                "matched_token_coverage": row.matched_token_coverage,
                "context_match_coverage": row.context_match_coverage,
                "top_collocate": row.top_collocate,
                "top_collocate_share": row.top_collocate_share,
            }
        )
audit_flags = pd.DataFrame(flag_rows)
audit_flags.to_csv(AUDIT_FLAGS_PATH, index=False)

pd.DataFrame(
    {
        "output": [
            ANNUAL_VALENCE_PATH.relative_to(PROJECT_ROOT),
            COVERAGE_PATH.relative_to(PROJECT_ROOT),
            TREND_SUMMARY_PATH.relative_to(PROJECT_ROOT),
            TOP_COLLOCATES_PATH.relative_to(PROJECT_ROOT),
            AUDIT_FLAGS_PATH.relative_to(PROJECT_ROOT),
            WARRINER_MATCHES_PATH.relative_to(PROJECT_ROOT),
        ]
    }
)


,output
0,data/processed/lsc/sentiment/robustness_warriner/lsc_sentiment_warriner_annual_valence.csv
1,data/processed/lsc/sentiment/robustness_warriner/lsc_sentiment_warriner_coverage.csv
2,data/processed/lsc/sentiment/robustness_warriner/lsc_sentiment_warriner_trend_models.csv
3,data/processed/lsc/sentiment/robustness_warriner/lsc_sentiment_warriner_top_collocates.csv
4,data/processed/lsc/sentiment/robustness_warriner/lsc_sentiment_warriner_audit_flags.csv
5,data/interim/lsc/warriner_vad/lsc_warriner_collocate_matches.parquet


## Handoff Summary

This compact check fails if the expected annual analysis grid is incomplete.


In [9]:
expected_annual_rows = len(EXPECTED_YEARS) * (len(BASELINE_UNITS) + len(TARGET_UNITS) * len(TARGET_FRAME_STRATA))
summary = {
    "annual_rows": len(annual_valence),
    "expected_annual_rows": expected_annual_rows,
    "warriner_match_rows": len(warriner_matches),
    "contexts_with_warriner_match": int(context_coverage["has_warriner_match"].sum()),
    "context_rows": len(context_coverage),
    "trend_rows": len(trend_summary),
    "audit_flag_rows": len(audit_flags),
    "comparison_rows": len(comparison),
}
if summary["annual_rows"] != expected_annual_rows:
    raise RuntimeError(f"Expected {expected_annual_rows} annual rows, found {summary['annual_rows']}.")
summary


{'annual_rows': 143,
 'expected_annual_rows': 143,
 'warriner_match_rows': 990407,
 'contexts_with_warriner_match': 306644,
 'context_rows': 311030,
 'trend_rows': 22,
 'audit_flag_rows': 1,
 'comparison_rows': 11}